In [1]:
pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 57.2 kB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
os.environ["ULTRALYTICS_NO_Ray"] = "True"
from ultralytics import YOLO, RTDETR
import numpy as np
from ultralytics.data.dataset import YOLODataset
from ultralytics.models.yolo.detect import train
import matplotlib.pyplot as plt
from glob import glob
import shutil
import cv2

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
import yaml

src_one_class = "/kaggle/input/datasets/mtoann/one-class-v5/final_dataset_split_5"
src_one_class_yaml = "/kaggle/input/datasets/mtoann/one-class-v5/final_dataset_split_5/data.yaml"
dst_one_class_yaml = "/kaggle/working/data.yaml"



src_yaml = "/kaggle/input/datasets/mtoann/damage-sign-v2/final_dataset_split_3/data.yaml"
dst_yaml = "/kaggle/working/data.yaml"
splits = ["train", "val", "test"]

# 1. One class detection only trainning

In [4]:
# for split in splits:
#     src_img = os.path.join(src_root, split, "images")
#     src_lbl = os.path.join(src_root, split, "labels")

#     dst_img = os.path.join(dst_root, split, "images")
#     dst_lbl = os.path.join(dst_root, split, "labels")

#     os.makedirs(dst_img, exist_ok=True)
#     os.makedirs(dst_lbl, exist_ok=True)

#     # Copy images
#     for img_file in glob(os.path.join(src_img, "*")):
#         shutil.copy(img_file, dst_img)

#     # Convert labels to single class
#     for lbl_file in glob(os.path.join(src_lbl, "*.txt")):
#         with open(lbl_file, "r") as f:
#             lines = f.readlines()

#         new_lines = []
#         for line in lines:
#             parts = line.strip().split()
#             if len(parts) >= 5:
#                 parts[0] = "0"  # 🔥 force class = 0
#                 new_lines.append(" ".join(parts))

#         with open(os.path.join(dst_lbl, os.path.basename(lbl_file)), "w") as f:
#             f.write("\n".join(new_lines))

# print("✅ One-class dataset created!")

In [5]:
with open(src_one_class_yaml, "r") as f:
    data = yaml.safe_load(f)

# KEEP path pointing to input (important!)
data["path"] = src_one_class

# Make sure splits are correct
data["train"] = "train/images"
data["val"]   = "val/images"
data["test"]  = "test/images"

with open(dst_one_class_yaml, "w") as f:
    yaml.dump(data, f)

print("New YAML saved at:", dst_one_class_yaml) 

New YAML saved at: /kaggle/working/data.yaml


In [6]:
# model = RTDETR("rtdetr-l.pt")

# # Train
# model.train(
#     data=dst_one_class_yaml,
#     epochs=30,          
#     imgsz=640,           # RT-DETR is native at 640; 800 may cause OOM on Kaggle
#     batch=16,            # RT-DETR uses significantly more VRAM than YOLO26s
    
#     optimizer="AdamW",   
#     lr0=0.0001,          # Lowered for Transformer stability
#     cos_lr=True,         
    
#     # RT-DETR usually performs better with lighter augmentation
#     mosaic=0.0,          # Often disabled for DETR models
#     mixup=0.0,
#     scale=0.5,           
    
#     workers=8,           
#     val=True,            
#     save=True,           
#     plots=True           
# )

In [7]:
# Load small pretrained model (VERY IMPORTANT)
model = YOLO("yolo26s.pt")

# Train
model.train(
    data=dst_one_class_yaml,
    epochs=100,
    imgsz=960,
    batch=32,

    optimizer="AdamW",
    lr0=0.001,
    cos_lr=True,

    mosaic=0.3,
    close_mosaic=10,
    mixup=0.1,
    copy_paste=0.2,
    scale=0.4,
    degrees=10,

    workers=8,

    overlap_mask=True,
    val=True,
    save=True,
    plots=True
)

Ultralytics 8.4.33 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.2, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=10, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=0.3, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=100, per

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7ae5ce5f72f0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [8]:
# Evaluate on test set
metrics = model.val(
    data=dst_one_class_yaml,
    split="test"
)

print(metrics)

Ultralytics 8.4.33 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
YOLO26s summary (fused): 122 layers, 9,465,567 parameters, 0 gradients, 20.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 12.7±3.9 MB/s, size: 67.1 KB)
val: Scanning /kaggle/input/datasets/mtoann/one-class-v5/final_dataset_split_5/test/labels... 451 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 451/451 226.6it/s 2.0s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/mtoann/one-class-v5/final_dataset_split_5/test is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 3.3it/s 8.7s
                   all        451        744      0.954      0.965      0.985      0.821
Speed: 3.3ms preprocess, 11.5ms inference, 0.0ms loss, 0.1ms postprocess per image
Results saved to /kaggle/working/runs/detect/val
ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([

# Multi-class detection

In [9]:
# with open(src_yaml, "r") as f:
#     data = yaml.safe_load(f)

# # ✅ KEEP path pointing to input (important!)
# data["path"] = "/kaggle/input/datasets/mtoann/damage-sign-v2/final_dataset_split_3"

# # Make sure splits are correct
# data["train"] = "train/images"
# data["val"]   = "val/images"
# data["test"]  = "test/images"

# with open(dst_yaml, "w") as f:
#     yaml.dump(data, f)

# print("New YAML saved at:", dst_yaml)

In [10]:
# with open(dst_yaml, "r") as f:
#     data = yaml.safe_load(f)

# base_path = data["path"]
# train_images = os.path.join(base_path, data["train"])
# train_labels = train_images.replace("images", "labels")
# class_names = data["names"]

# samples_per_class = 2
# class_samples = {i: [] for i in range(len(class_names))}

# def yolo_to_xyxy(img_w, img_h, x, y, w, h):
#     x1 = int((x - w / 2) * img_w)
#     y1 = int((y - h / 2) * img_h)
#     x2 = int((x + w / 2) * img_w)
#     y2 = int((y + h / 2) * img_h)
#     return x1, y1, x2, y2

# for label_file in os.listdir(train_labels):
#     label_path = os.path.join(train_labels, label_file)

#     with open(label_path, "r") as f:
#         lines = f.readlines()

#     for line in lines:
#         parts = line.strip().split()
#         class_id = int(parts[0])

#         if len(class_samples[class_id]) < samples_per_class:
#             img_file = label_file.replace(".txt", ".jpg")
#             img_path = os.path.join(train_images, img_file)

#             if os.path.exists(img_path):
#                 class_samples[class_id].append((img_path, label_path))

# fig, axes = plt.subplots(len(class_names), samples_per_class, figsize=(12, 10))
# for class_id, samples in class_samples.items():
#     for i in range(samples_per_class):
#         ax = axes[class_id, i] if len(class_names) > 1 else axes[i]

#         if i < len(samples):
#             img_path, label_path = samples[i]

#             img = cv2.imread(img_path)
#             h, w, _ = img.shape

#             # Draw boxes
#             with open(label_path, "r") as f:
#                 for line in f.readlines():
#                     c, x, y, bw, bh = map(float, line.strip().split())
#                     c = int(c)

#                     x1, y1, x2, y2 = yolo_to_xyxy(w, h, x, y, bw, bh)

#                     # Draw rectangle
#                     cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)

#                     # Put label
#                     cv2.putText(
#                         img,
#                         class_names[c],
#                         (x1, y1 - 5),
#                         cv2.FONT_HERSHEY_SIMPLEX,
#                         0.5,
#                         (0, 255, 0),
#                         1,
#                         cv2.LINE_AA,
#                     )

#             img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
#             ax.imshow(img)
#             ax.set_title(class_names[class_id])

#         ax.axis("off")

# plt.tight_layout()
# plt.show()

In [11]:
# class YOLOWeightedDataset(YOLODataset):
#     def __init__(self, *args, **kwargs):
#         super().__init__(*args, **kwargs)

#         # Detect train mode
#         self.train_mode = self.prefix == "train"
#         # Count instances per class
#         self.count_instances()
#         # Compute class weights (inverse frequency)
#         self.class_weights = np.sum(self.counts) / (self.counts + 1e-6)
#         # Aggregation strategy (IMPORTANT)
#         self.agg_func = np.max   # better than mean for detection
#         # Compute image weights
#         self.weights = self.calculate_weights()
#         self.probabilities = self.calculate_probabilities()

#     def count_instances(self):
#         self.counts = [0 for _ in range(len(self.data["names"]))]
#         for label in self.labels:
#             cls = label['cls'].reshape(-1).astype(int)
#             for c in cls:
#                 self.counts[c] += 1
#         self.counts = np.array(self.counts)
#         self.counts = np.where(self.counts == 0, 1, self.counts)

#     def calculate_weights(self):
#         weights = []
#         for label in self.labels:
#             cls = label['cls'].reshape(-1).astype(int)

#             if cls.size == 0:
#                 weights.append(1.0)
#                 continue

#             weight = self.agg_func(self.class_weights[cls])
#             weights.append(weight)

#         return weights

#     def calculate_probabilities(self):
#         total = sum(self.weights)
#         return [w / total for w in self.weights]

#     def __getitem__(self, index):
#         if not self.train_mode:
#             return self.transforms(self.get_image_and_label(index))
#         else:
#             index = np.random.choice(len(self.labels), p=self.probabilities)
#             return self.transforms(self.get_image_and_label(index))

#     def plot_distributions(self):    
#         class_names = self.data["names"]
    
#         # ---------- BEFORE (raw counts) ----------
#         raw_counts = self.counts
#         raw_distribution = raw_counts / raw_counts.sum()
    
#         # ---------- AFTER (weighted sampling) ----------
#         sampled_counts = np.zeros(len(class_names))
    
#         # simulate sampling
#         num_samples = 5000
#         for _ in range(num_samples):
#             idx = np.random.choice(len(self.labels), p=self.probabilities)
#             cls = self.labels[idx]['cls'].reshape(-1).astype(int)
#             for c in cls:
#                 sampled_counts[c] += 1
    
#         sampled_distribution = sampled_counts / sampled_counts.sum()
    
#         # ---------- PLOT ----------
#         x = np.arange(len(class_names))
    
#         plt.figure(figsize=(10, 5))
#         plt.bar(x - 0.2, raw_distribution, width=0.4, label="Before (Raw)")
#         plt.bar(x + 0.2, sampled_distribution, width=0.4, label="After (Weighted)")
    
#         plt.xticks(x, class_names)
#         plt.ylabel("Distribution")
#         plt.title("Class Distribution Before vs After Weighting")
#         plt.legend()
    
#         plt.show()


# train.YOLODataset = YOLOWeightedDataset
# dataset = YOLOWeightedDataset(
#     img_path="/kaggle/input/datasets/mtoann/damage-sign/final_dataset_split_2/train/images",
#     data={"names": ["informational_guide", "regulatory", "warning"]},
#     task="detect"
# )

# dataset.plot_distributions()

In [12]:
# # Load small pretrained model (VERY IMPORTANT)
# model = YOLO("yolo26n.pt")  # nano = fastest & lightweight

# # Train
# model.train(
#     data="/kaggle/working/data.yaml",
#     epochs=100,
#     imgsz=640,
#     batch=16,

#     lr0=0.01,
#     optimizer="SGD",

#     # Augmentation
#     hsv_h=0.015,
#     hsv_s=0.7,
#     hsv_v=0.4,
#     fliplr=0.5,
#     mosaic=1.0,
#     mixup=0.2,
#     copy_paste=0.3,

#     project="runs",
#     name="yolo_weighted",
# )



In [13]:
# # Evaluate on test set
# metrics = model.val(
#     data="/kaggle/working/data.yaml",
#     split="test"
# )

# print(metrics)